In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# load files (reuse variable names if re-executed)
df_metric = pd.read_csv('metric.csv', low_memory=False)
df_trace = pd.read_csv('trace.csv', low_memory=False)
df_log = pd.read_csv('log.csv', low_memory=False)
df_error = pd.read_csv('error_logs.csv', low_memory=False)

# helper to produce compact timestamp range info (ints + Asia/Shanghai ISO)
def ts_range_info(ts_series):
    if ts_series is None or ts_series.dropna().empty:
        return {'min_ts': None, 'max_ts': None, 'min_dt_shanghai': None, 'max_dt_shanghai': None}
    min_ts = int(ts_series.min())
    max_ts = int(ts_series.max())
    # convert via UTC then to Asia/Shanghai
    min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    return {'min_ts': min_ts, 'max_ts': max_ts, 'min_dt_shanghai': min_dt, 'max_dt_shanghai': max_dt}

# compact unique listing (limit to first 20)
def unique_limited(series, limit=20):
    if series is None:
        return []
    return list(pd.Index(series.dropna().unique()[:limit]))

# summaries
metric_summary = {}
metric_summary['total_rows'] = int(df_metric.shape[0])
metric_summary.update(ts_range_info(df_metric['timestamp'] if 'timestamp' in df_metric.columns else None))
metric_summary['unique_cmdb_id_sample'] = unique_limited(df_metric['cmdb_id'])
metric_summary['unique_kpi_name_sample'] = unique_limited(df_metric['kpi_name'])
# top 20 (cmdb_id, kpi_name) pairs with counts
if {'cmdb_id','kpi_name'}.issubset(df_metric.columns):
    top_pairs_metric = (
        df_metric.groupby(['cmdb_id','kpi_name']).size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_pairs_metric = pd.DataFrame(columns=['cmdb_id','kpi_name','count'])
metric_summary['top_20_cmdbid_kpi_pairs'] = top_pairs_metric

trace_summary = {}
trace_summary['total_rows'] = int(df_trace.shape[0])
trace_summary.update(ts_range_info(df_trace['timestamp'] if 'timestamp' in df_trace.columns else None))
trace_summary['unique_cmdb_id_sample'] = unique_limited(df_trace['cmdb_id'])
trace_summary['unique_trace_name_sample'] = unique_limited(df_trace['trace_name'])
if {'cmdb_id','trace_name'}.issubset(df_trace.columns):
    top_pairs_trace = (
        df_trace.groupby(['cmdb_id','trace_name']).size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_pairs_trace = pd.DataFrame(columns=['cmdb_id','trace_name','count'])
trace_summary['top_20_cmdbid_trace_pairs'] = top_pairs_trace

log_summary = {}
log_summary['total_rows'] = int(df_log.shape[0])
log_summary.update(ts_range_info(df_log['timestamp'] if 'timestamp' in df_log.columns else None))
log_summary['unique_cmdb_id_sample'] = unique_limited(df_log['cmdb_id'])
log_summary['unique_log_name_sample'] = unique_limited(df_log['log_name'])
if {'cmdb_id','log_name'}.issubset(df_log.columns):
    top_pairs_log = (
        df_log.groupby(['cmdb_id','log_name']).size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_pairs_log = pd.DataFrame(columns=['cmdb_id','log_name','count'])
log_summary['top_20_cmdbid_log_pairs'] = top_pairs_log

error_summary = {}
error_summary['total_rows'] = int(df_error.shape[0])
error_summary.update(ts_range_info(df_error['timestamp'] if 'timestamp' in df_error.columns else None))
# up to 20 distinct (timestamp, cmdb_id) samples
if {'timestamp','cmdb_id'}.issubset(df_error.columns):
    samples = df_error[['timestamp','cmdb_id']].drop_duplicates().head(20).reset_index(drop=True)
else:
    samples = pd.DataFrame(columns=['timestamp','cmdb_id'])
error_summary['distinct_timestamp_cmdb_samples_up_to_20'] = samples

# final compact structured summary
summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_summary
}

summary

```
Out[1]:
```
result_summary = (
    "Telemetry summary (concise):\n\n"
    "metric.csv: 320,427 rows. Time range 2022-03-20 20:30 -> 21:30 (Asia/Shanghai). "
    "Sample cmdb_id values (first 20): adservice, adservice-0, adservice-1, adservice-2, adservice2, adservice2-0, cartservice, cartservice-0, cartservice-1, cartservice-2, cartservice2-0, checkoutservice, checkoutservice-0, checkoutservice-1, checkoutservice-2, checkoutservice2-0, currencyservice, currencyservice-0, currencyservice-1, currencyservice-2. "
    "Sample kpi_names (first 20) include app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count, runtime.java_lang_ClassLoading_..., etc. "
    "Top (cmdb_id, kpi_name) pairs are dominated by shippingservice2-0 entries; top pair counts are 61 (top 20 pairs shown).\n\n"
    "trace.csv: 31,840 rows. Time range 2022-03-20 21:00 -> 21:29 (Asia/Shanghai). "
    "Sample cmdb_ids include adservice-0..2, cartservice-0..2, checkoutservice-0..2, currencyservice-0..2, emailservice-0..2, etc. "
    "Sample trace_names include trace.from_frontend-0.duration_mean/p95/error_rate/row_count, trace.from_frontend-1.*, trace.from_frontend2-0.*, etc. "
    "Top (cmdb_id, trace_name) pairs have counts = 30 (top entries include shippingservice2-0 and adservice-0 among others).\n\n"
    "log.csv: 1,816 rows. Time range 2022-03-20 21:00 -> 21:30 (Asia/Shanghai). "
    "Sample cmdb_ids include frontend-0..2, paymentservice-0..1, currencyservice-0..2, cartservice-0..2, etc. "
    "Unique log_name values: log.error_count, log.row_count. "
    "Top (cmdb_id, log_name) pairs (top 20) largely show cartservice-*, currencyservice-*, frontend-0, recommendationservice-* with counts = 31.\n\n"
    "error_logs.csv: 16 rows. Time range 2022-03-20 21:17:32 -> 21:21:05 (Asia/Shanghai). "
    "Up to 20 distinct (timestamp, cmdb_id) samples include frontend-1 and checkoutservice-1/2/0 at various timestamps (multiple entries between 21:17:32 and 21:21:05)."
)
result_summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 320427, 'min_ts': 1647779400, 'max_ts': 1647783000, 'min_dt_shanghai': '2022-03-20T20:30:00+08:00', 'max_dt_shanghai': '2022-03-20T21:30:00+08:00', 'unique_cmdb_id_sample': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'unique_kpi_name_sample': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact'], 'top_20_cmdbid_kpi_pairs':                cmdb_id                                           kpi_name  count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61
1   shippingservice2-0  container.node-6.container_network_receive_MB....     61
2   shippingservice2-0  container.node-6.container_network_receive_err...     61
3   shippingservice2-0  container.node-6.container_network_receive_pac...     61
4   shippingservice2-0  container.node-6.container_network_receive_pac...     61
5   shippingservice2-0  container.node-6.container_network_transmit_MB...     61
6   shippingservice2-0  container.node-6.container_network_transmit_er...     61
7   shippingservice2-0  container.node-6.container_network_transmit_pa...     61
8   shippingservice2-0  container.node-6.container_network_transmit_pa...     61
9   shippingservice2-0  container.node-6.container_memory_failures.hie...     61
10  shippingservice2-0  container.node-6.container_memory_failures.hie...     61
11  shippingservice2-0      container.node-6.container_memory_mapped_file     61
12  shippingservice2-0     container.node-6.container_memory_max_usage_MB     61
13  shippingservice2-0              container.node-6.container_memory_rss     61
14  shippingservice2-0             container.node-6.container_memory_swap     61
15  shippingservice2-0         container.node-6.container_memory_usage_MB     61
16  shippingservice2-0   container.node-6.container_memory_working_set_MB     61
17  shippingservice2-0     container.node-6.container_fs_writes./dev/vda1     61
18  shippingservice2-0   container.node-6.container_fs_writes_MB./dev/vda     61
19  shippingservice2-0  container.node-6.container_fs_writes_merged./d...     61}, 'trace.csv': {'total_rows': 31840, 'min_ts': 1647781200, 'max_ts': 1647782940, 'min_dt_shanghai': '2022-03-20T21:00:00+08:00', 'max_dt_shanghai': '2022-03-20T21:29:00+08:00', 'unique_cmdb_id_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'unique_trace_name_sample': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count'], 'top_20_cmdbid_trace_pairs':                cmdb_id                            trace_name  count
0   shippingservice2-0      trace.from_frontend2-0.row_count     30
1          adservice-0   trace.from_frontend-0.duration_mean     30
2          adservice-0    trace.from_frontend-0.duration_p95     30
3          adservice-0      trace.from_frontend-0.error_rate     30
4          adservice-0       trace.from_frontend-0.row_count     30
5          adservice-0   trace.from_frontend-1.duration_mean     30
6          adservice-0    trace.from_frontend-1.duration_p95     30
7          adservice-0      trace.from_frontend-1.error_rate     30
8          adservice-0       trace.from_frontend-1.row_count     30
9   shippingservice2-0     trace.from_frontend2-0.error_rate     30
10  shippingservice2-0   trace.from_frontend2-0.duration_p95     30
11  shippingservice2-0  trace.from_frontend2-0.duration_mean     30
12   shippingservice-1   trace.from_frontend-1.duration_mean     30
13   shippingservice-1    trace.from_frontend-1.duration_p95     30
14   shippingservice-1      trace.from_frontend-1.error_rate     30
15   shippingservice-1       trace.from_frontend-1.row_count     30
16   shippingservice-0   trace.from_frontend-1.duration_mean     30
17   shippingservice-0    trace.from_frontend-1.duration_p95     30
18   shippingservice-0      trace.from_frontend-1.error_rate     30
19   shippingservice-0       trace.from_frontend-1.row_count     30}, 'log.csv': {'total_rows': 1816, 'min_ts': 1647781200, 'max_ts': 1647783000, 'min_dt_shanghai': '2022-03-20T21:00:00+08:00', 'max_dt_shanghai': '2022-03-20T21:30:00+08:00', 'unique_cmdb_id_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'unique_log_name_sample': ['log.error_count', 'log.row_count'], 'top_20_cmdbid_log_pairs':                     cmdb_id         log_name  count
0             cartservice-2    log.row_count     31
1             cartservice-2  log.error_count     31
2             cartservice-1    log.row_count     31
3             cartservice-1  log.error_count     31
4             cartservice-0    log.row_count     31
5             cartservice-0  log.error_count     31
6         currencyservice-0  log.error_count     31
7         currencyservice-0    log.row_count     31
8         currencyservice-2  log.error_count     31
9         currencyservice-2    log.row_count     31
10        currencyservice-1  log.error_count     31
11        currencyservice-1    log.row_count     31
12               frontend-0    log.row_count     31
13               frontend-0  log.error_count     31
14  recommendationservice-0    log.row_count     31
15  recommendationservice-0  log.error_count     31
16  recommendationservice-1    log.row_count     31
17  recommendationservice-1  log.error_count     31
18  recommendationservice-2  log.error_count     31
19  recommendationservice-2    log.row_count     31}, 'error_logs.csv': {'total_rows': 16, 'min_ts': 1647782252, 'max_ts': 1647782465, 'min_dt_shanghai': '2022-03-20T21:17:32+08:00', 'max_dt_shanghai': '2022-03-20T21:21:05+08:00', 'distinct_timestamp_cmdb_samples_up_to_20':      timestamp            cmdb_id
0   1647782252         frontend-1
1   1647782252  checkoutservice-1
2   1647782258         frontend-2
3   1647782258  checkoutservice-1
4   1647782281         frontend-1
5   1647782281  checkoutservice-2
6   1647782346         frontend-1
7   1647782346  checkoutservice-2
8   1647782400  checkoutservice-1
9   1647782400         frontend-1
10  1647782407         frontend-1
11  1647782407  checkoutservice-0
12  1647782415  checkoutservice-1
13  1647782415         frontend-0
14  1647782465  checkoutservice-1
15  1647782465         frontend-1}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse or load metric data
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

# ensure correct dtypes
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Global statistics per (cmdb_id, kpi_name) over full series
group_keys = ['cmdb_id', 'kpi_name']
agg_funcs = {
    'global_count': ('value', 'count'),
    'global_min': ('value', 'min'),
    'global_max': ('value', 'max'),
    'global_mean': ('value', 'mean'),
    'global_P95': (lambda s: s.quantile(0.95)),
    'global_P90': (lambda s: s.quantile(0.90)),
    'global_P5':  (lambda s: s.quantile(0.05)),
}

# Using a custom groupby to compute quantiles along with other aggs
gb = df_metric.groupby(group_keys)['value']
global_stats_df = gb.agg([
    ('global_count', 'count'),
    ('global_min', 'min'),
    ('global_max', 'max'),
    ('global_mean', 'mean'),
    ('global_P95', lambda x: x.quantile(0.95)),
    ('global_P90', lambda x: x.quantile(0.90)),
    ('global_P5',  lambda x: x.quantile(0.05))
]).reset_index()

# store full global table in kernel memory as required
global_stats = global_stats_df  # kept for further joins

# 2) Filter metric.csv to incident window [1647781200, 1647783000]
start_ts = 1647781200
end_ts = 1647783000
window_df = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Windowed stats per pair: count_in_window, min_in_window, max_in_window, earliest_timestamp_in_window
window_grp = window_df.groupby(group_keys)
window_stats_df = window_grp.agg(
    count_in_window=('value','count'),
    min_in_window=('value','min'),
    max_in_window=('value','max'),
    earliest_timestamp_in_window=('timestamp','min')
).reset_index()

# 4) Mark anomalies in the window using global thresholds (P95, P5)
# Merge global thresholds into window rows
window_with_thresholds = window_df.merge(
    global_stats_df[['cmdb_id','kpi_name','global_P95','global_P5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Anomaly condition: value >= global_P95 OR value <= global_P5
window_with_thresholds['is_anomaly'] = (
    (window_with_thresholds['value'] >= window_with_thresholds['global_P95']) |
    (window_with_thresholds['value'] <= window_with_thresholds['global_P5'])
)

# 5) Per-pair anomaly counts and earliest anomaly timestamp
anomaly_grp = window_with_thresholds[window_with_thresholds['is_anomaly']].groupby(group_keys)
anomaly_agg = anomaly_grp.agg(
    anomaly_count_in_window=('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window=('timestamp','min')
).reset_index()

# Merge window_stats and global_stats and anomaly_agg to compute severity
agg_df = global_stats_df.merge(window_stats_df, on=group_keys, how='left')
agg_df = agg_df.merge(anomaly_agg, on=group_keys, how='left')

# fill NaNs for counts with 0 where appropriate
agg_df['count_in_window'] = agg_df['count_in_window'].fillna(0).astype(int)
agg_df['anomaly_count_in_window'] = agg_df['anomaly_count_in_window'].fillna(0).astype(int)
agg_df['earliest_anomaly_timestamp_in_window'] = agg_df['earliest_anomaly_timestamp_in_window'].where(
    ~agg_df['earliest_anomaly_timestamp_in_window'].isna(), None)

# compute severity per instruction
# severity = max((max_in_window - P95)/P95 if P95>0 else 0, (P5 - min_in_window)/P5 if P5>0 else 0)
def compute_severity(row):
    p95 = row['global_P95']
    p5 = row['global_P5']
    max_w = row['max_in_window'] if not pd.isna(row['max_in_window']) else row['global_max']
    min_w = row['min_in_window'] if not pd.isna(row['min_in_window']) else row['global_min']
    a = (max_w - p95) / p95 if (p95 is not None and p95 > 0) else 0
    b = (p5 - min_w) / p5 if (p5 is not None and p5 > 0) else 0
    return float(max(a, b, 0.0))

agg_df['severity'] = agg_df.apply(compute_severity, axis=1)

# Convert earliest anomaly ts to Asia/Shanghai ISO strings for display (or None)
def to_shanghai_iso(ts):
    if ts is None or pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

agg_df['earliest_anomaly_ts_shanghai'] = agg_df['earliest_anomaly_timestamp_in_window'].apply(to_shanghai_iso)
agg_df['earliest_window_ts_shanghai'] = agg_df['earliest_timestamp_in_window'].apply(
    lambda x: to_shanghai_iso(x) if not pd.isna(x) else None
)

# 6) Prepare final table and top 20 rows sorted by anomaly_count desc, severity desc
final_cols = [
    'cmdb_id','kpi_name',
    'global_count','global_P95','global_P5',
    'count_in_window','anomaly_count_in_window','earliest_anomaly_ts_shanghai',
    'min_in_window','max_in_window','severity'
]

# ensure columns exist in agg_df (some window stats may be missing -> keep as NaN/None)
for c in final_cols:
    if c not in agg_df.columns:
        agg_df[c] = None

top20 = agg_df[final_cols].sort_values(
    by=['anomaly_count_in_window','severity'], ascending=[False, False]
).head(20).reset_index(drop=True)

# total distinct pairs processed (based on global stats)
total_pairs_count = int(global_stats_df.shape[0])

# Keep full intermediate tables in kernel memory:
global_stats_df, window_df, window_stats_df, window_with_thresholds, anomaly_agg, agg_df

# Display compact results: top20 table and total count
top20, total_pairs_count

```
Out[2]:
```
# Plain English summary based on the computed results
top5 = top20.head(5)[['cmdb_id','kpi_name','anomaly_count_in_window','severity']]
total_pairs = total_pairs_count

summary = (
    f"Summary of metric anomaly analysis (incident window 2022-03-20 21:00:00 to 2022-03-20 21:30:00 Asia/Shanghai):\n\n"
    f"- Total distinct (cmdb_id, kpi_name) pairs processed: {total_pairs}\n"
    f"- Window duration: 31 minutes (31 samples per pair when present in window).\n"
    f"- Top 20 results (shown earlier) all have anomaly_count_in_window = 31, i.e., every sample in the window was flagged anomalous for those pairs.\n"
    f"- Earliest anomaly for the top pairs is at the start of the window: 2022-03-20T21:00:00+08:00.\n\n"
    f"- Top 5 anomalous pairs (cmdb_id, kpi_name) with anomaly count and severity:\n"
)

for i, row in top5.iterrows():
    summary += f"  {i+1}. {row['cmdb_id']} | {row['kpi_name']} — anomalies: {int(row['anomaly_count_in_window'])}, severity: {row['severity']:.6g}\n"

summary += (
    "\nNotes:\n"
    "- Anomaly rule: value >= global P95 OR value <= global P5 (global quantiles computed over the full series).\n"
    "- Many global P5 values are 0, and several global P95 values are very small; this produced large severity scores where window maxima greatly exceeded global P95.\n"
    "- The top-ranked anomalies include filesystem read metrics, I/O and CPU wait metrics, and several container/node-level and mesh metrics for checkoutservice and nodes.\n"
    "- These findings indicate sustained deviations across the entire 31-minute window for the top pairs; investigate the listed components (e.g., checkoutservice-2, node-4/5, related container metrics) first.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name  global_count    global_P95   global_P5  count_in_window  anomaly_count_in_window earliest_anomaly_ts_shanghai  min_in_window  max_in_window   severity
0          checkoutservice-2    container.node-5.container_fs_reads_MB./dev/vda            61  1.171875e-02         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   2.617188e-01  21.333333
1          checkoutservice-2       container.node-5.container_fs_reads./dev/vda            61  2.000000e+00         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   2.200000e+01  10.000000
2                     node-4                                    system.io.svctm            61  7.000000e-02         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   5.000000e-01   6.142857
3                     node-5                                  system.cpu.iowait            61  2.000000e-02         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.300000e-01   5.500000
4   recommendationservice2-0  container.node-6.container_cpu_cfs_throttled_s...            61  2.194960e-02         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   7.100902e-02   2.235094
5             emailservice-2  mesh.source.emailservice.jaeger-collector.isti...            61  5.250000e-04         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.575000e-03   2.000000
6              cartservice-0            container.node-6.container_memory_cache            61  1.635533e+07  16355328.0               31                       31    2022-03-20T21:00:00+08:00     16355328.0   3.717120e+07   1.272727
7          checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...            61  5.950000e+01         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.210000e+02   1.033613
8           paymentservice-0      container.node-6.container_cpu_system_seconds            61  5.000000e-03         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.000000e-02   1.000000
9          shippingservice-0  mesh.destination.checkoutservice.shippingservi...            61  5.250000e-04         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.050000e-03   1.000000
10          paymentservice-2      container.node-6.container_cpu_system_seconds            61  5.000000e-03         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.000000e-02   1.000000
11         checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...            61  7.850000e+02         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.570000e+03   1.000000
12         checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...            61  1.000000e+00         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   2.000000e+00   1.000000
13         checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...            61  1.850000e+03         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   3.700000e+03   1.000000
14         checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...            61  5.950000e+01         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.190000e+02   1.000000
15         checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...            61  1.000000e+00         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   2.000000e+00   1.000000
16         currencyservice-0  container.node-6.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.000000e+00   1.000000
17         currencyservice-1  container.node-6.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.000000e+00   1.000000
18         currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...            61  5.000000e-01         0.0               31                       31    2022-03-20T21:00:00+08:00            0.0   1.000000e+00   1.000000
19  productcatalogservice2-0                 container.node-6.container_sockets            61  3.500000e+00         3.0               31                       31    2022-03-20T21:00:00+08:00            3.0   4.500000e+00   0.285714, 5281)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)

# ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')
group_keys = ['cmdb_id', 'trace_name']

# 1) Global statistics over full series (count, P95, P90, P5)
gb_trace = df_trace.groupby(group_keys)['value']
global_stats_trace = gb_trace.agg(
    global_count=('count'),
    global_P95=(lambda x: x.quantile(0.95)),
    global_P90=(lambda x: x.quantile(0.90)),
    global_P5=(lambda x: x.quantile(0.05))
).reset_index()

# 2) Filter to incident window [1647781200, 1647783000]
start_ts = 1647781200
end_ts = 1647783000
window_trace_df = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Windowed stats per (cmdb_id, trace_name)
window_stats_trace = window_trace_df.groupby(group_keys).agg(
    count_in_window=('value','count'),
    min_in_window=('value','min'),
    max_in_window=('value','max'),
    earliest_timestamp_in_window=('timestamp','min')
).reset_index()

# 4) Mark anomalies in window rows using global thresholds (P95, P5)
window_with_thresholds_trace = window_trace_df.merge(
    global_stats_trace[['cmdb_id','trace_name','global_P95','global_P5']],
    on=['cmdb_id','trace_name'],
    how='left'
)
window_with_thresholds_trace['is_anomaly'] = (
    (window_with_thresholds_trace['value'] >= window_with_thresholds_trace['global_P95']) |
    (window_with_thresholds_trace['value'] <= window_with_thresholds_trace['global_P5'])
)

# 5) Per-pair anomaly counts and earliest anomaly timestamp
anomaly_grp_trace = window_with_thresholds_trace[window_with_thresholds_trace['is_anomaly']].groupby(group_keys)
anomaly_agg_trace = anomaly_grp_trace.agg(
    anomaly_count_in_window=('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window=('timestamp','min')
).reset_index()

# Merge globals, window stats, anomaly stats
agg_trace_df = global_stats_trace.merge(window_stats_trace, on=group_keys, how='left')
agg_trace_df = agg_trace_df.merge(anomaly_agg_trace, on=group_keys, how='left')

# Fill missing counts with 0 and ensure types
agg_trace_df['count_in_window'] = agg_trace_df['count_in_window'].fillna(0).astype(int)
agg_trace_df['anomaly_count_in_window'] = agg_trace_df['anomaly_count_in_window'].fillna(0).astype(int)
agg_trace_df['earliest_anomaly_timestamp_in_window'] = agg_trace_df['earliest_anomaly_timestamp_in_window'].where(
    ~agg_trace_df['earliest_anomaly_timestamp_in_window'].isna(), None)

# 5b) Compute severity
def compute_severity_trace(row):
    p95 = row['global_P95']
    p5 = row['global_P5']
    max_w = row['max_in_window'] if not pd.isna(row.get('max_in_window', np.nan)) else np.nan
    min_w = row['min_in_window'] if not pd.isna(row.get('min_in_window', np.nan)) else np.nan
    # if no window values, severity 0
    if pd.isna(max_w) and pd.isna(min_w):
        return 0.0
    a = (max_w - p95) / p95 if (pd.notna(p95) and p95 > 0 and pd.notna(max_w)) else 0.0
    b = (p5 - min_w) / p5 if (pd.notna(p5) and p5 > 0 and pd.notna(min_w)) else 0.0
    return float(max(a, b, 0.0))

agg_trace_df['severity'] = agg_trace_df.apply(compute_severity_trace, axis=1)

# Convert earliest anomaly ts to Asia/Shanghai ISO strings for display (or None)
def to_shanghai_iso(ts):
    if ts is None or pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

agg_trace_df['earliest_anomaly_ts_shanghai'] = agg_trace_df['earliest_anomaly_timestamp_in_window'].apply(to_shanghai_iso)
agg_trace_df['earliest_window_ts_shanghai'] = agg_trace_df['earliest_timestamp_in_window'].apply(
    lambda x: to_shanghai_iso(x) if not pd.isna(x) else None
)

# 6) Prepare final top-20 table
final_cols = [
    'cmdb_id','trace_name',
    'global_count','global_P95','global_P5',
    'count_in_window','anomaly_count_in_window','earliest_anomaly_ts_shanghai',
    'min_in_window','max_in_window','severity'
]
# ensure columns exist
for c in final_cols:
    if c not in agg_trace_df.columns:
        agg_trace_df[c] = None

top20_trace = agg_trace_df[final_cols].sort_values(
    by=['anomaly_count_in_window','severity'], ascending=[False, False]
).head(20).reset_index(drop=True)

total_pairs_trace = int(global_stats_trace.shape[0])

# Keep full intermediate tables in kernel memory:
# global_stats_trace, window_trace_df, window_stats_trace, window_with_thresholds_trace, anomaly_agg_trace, agg_trace_df

# Display compact results: top20_trace and total_pairs_trace
top20_trace, total_pairs_trace

```
Out[3]:
```
summary = (
    "Trace-level anomaly scan (incident window 2022-03-20 21:00–21:29 Asia/Shanghai):\n\n"
    "- Total distinct (cmdb_id, trace_name) pairs processed: 1,232.\n"
    "- Window length produced 30 samples per pair when present; top candidates all have anomaly_count_in_window = 30 (i.e., every sample in-window was flagged).\n"
    "- Highest-severity hits: \n"
    "  * cartservice2-0 | trace.from_frontend2-0.duration_p95 — global_P95=0.001, window max=9.408, anomaly_count=30, severity≈9407.0\n"
    "  * frontend2-0   | trace.to_cartservice2-0.duration_p95  — global_P95=0.001, window max=9.408, anomaly_count=30, severity≈9407.0\n"
    "- Many other top entries are trace.error_rate series (e.g., adservice-*, cartservice-*) where global P95 and P5 equal 0.0. Those series were flagged (anomaly_count=30) because values met the >=P95 or <=P5 rule, but their computed severity is 0.0 — these are likely threshold/equality artifacts rather than large magnitude spikes.\n"
    "- Earliest anomaly for top candidates is at the start of the window: 2022-03-20T21:00:00+08:00.\n\n"
    "Actionable guidance: prioritize investigation of the two high-severity duration spikes (cartservice2-0 and frontend2-0 traces to cartservice2-0). Treat the numerous zero-valued error_rate hits with caution (possible false positives due to zero quantiles); verify thresholds or inspect raw trace values before acting on them."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                            trace_name  global_count  global_P95  global_P5  count_in_window  anomaly_count_in_window earliest_anomaly_ts_shanghai  min_in_window  max_in_window  severity
0   cartservice2-0   trace.from_frontend2-0.duration_p95            30       0.001      0.001               30                       30    2022-03-20T21:00:00+08:00          0.001          9.408    9407.0
1      frontend2-0  trace.to_cartservice2-0.duration_p95            30       0.001      0.001               30                       30    2022-03-20T21:00:00+08:00          0.001          9.408    9407.0
2      adservice-0      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
3      adservice-0      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
4      adservice-1      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
5      adservice-1      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
6      adservice-2      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
7      adservice-2      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
8     adservice2-0     trace.from_frontend2-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
9    cartservice-0      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
10   cartservice-0      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
11   cartservice-0      trace.from_frontend-2.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
12   cartservice-0               trace.self.duration_p95            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
13   cartservice-0                 trace.self.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
14   cartservice-1      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
15   cartservice-1      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
16   cartservice-1               trace.self.duration_p95            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
17   cartservice-1                 trace.self.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
18   cartservice-2      trace.from_frontend-0.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0
19   cartservice-2      trace.from_frontend-1.error_rate            30       0.000      0.000               30                       30    2022-03-20T21:00:00+08:00          0.000          0.000       0.0, 1232)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing dataframes if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv', low_memory=False)

try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv', low_memory=False)

# ensure types
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

df_error['timestamp'] = df_error['timestamp'].astype(int)

# timezone
tz = pytz.timezone('Asia/Shanghai')

# incident window
start_ts = 1647781200
end_ts = 1647783000

# 1) log.csv analysis: filter to window and aggregate per (cmdb_id, log_name)
log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

log_agg = (
    log_window.groupby(['cmdb_id','log_name'])
    .agg(
        count_of_rows_in_window=('value','count'),
        sum_value_in_window=('value','sum'),
        max_value_in_window=('value','max'),
        earliest_timestamp_in_window=('timestamp','min')
    )
    .reset_index()
)

# convert earliest timestamp to Asia/Shanghai ISO for compact display
def to_shanghai_iso(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

log_agg['earliest_ts_shanghai'] = log_agg['earliest_timestamp_in_window'].apply(to_shanghai_iso)

# Filter for log_name == 'log.error_count' and take top 20 by sum_value_in_window desc
top_log_error_count = (
    log_agg[log_agg['log_name'] == 'log.error_count']
    .sort_values(by='sum_value_in_window', ascending=False)
    .head(20)
    .loc[:, ['cmdb_id','count_of_rows_in_window','sum_value_in_window','max_value_in_window','earliest_ts_shanghai']]
    .reset_index(drop=True)
)

# 2) error_logs.csv analysis: filter to window, distinct rows, up to 20 ordered by timestamp asc
error_window = df_error[(df_error['timestamp'] >= start_ts) & (df_error['timestamp'] <= end_ts)].copy()

# keep distinct (timestamp, cmdb_id, message) and order by timestamp asc, limit 20
error_samples = (
    error_window.drop_duplicates(subset=['timestamp','cmdb_id','message'])
    .sort_values('timestamp', ascending=True)
    .head(20)
    .loc[:, ['timestamp','cmdb_id','message']]
    .reset_index(drop=True)
)

# Also provide counts per cmdb_id in error logs within the window (top 10)
error_counts_by_cmdb = (
    error_window.groupby('cmdb_id').size().reset_index(name='error_count_in_window')
    .sort_values('error_count_in_window', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Keep full intermediates in kernel memory (variables defined above)
log_window, log_agg, error_window

# Return compact results: top_log_error_count (<=20 rows), error_samples (<=20 rows), error_counts_by_cmdb (top 10)
top_log_error_count, error_samples, error_counts_by_cmdb

```
Out[4]:
```
Summary of log and error-log analysis for incident window 2022-03-20 21:00:00 → 21:30:00 (Asia/Shanghai)

1) log.csv — aggregated (log_name == "log.error_count", top 20 by sum_value_in_window)
- Top offenders (cmdb_id | rows_in_window | sum_error_count | max_value | earliest_timestamp):
  1. frontend-1 | 31 rows | sum=6.0 | max=2.0 | earliest=2022-03-20T21:00:00+08:00  
  2. checkoutservice-1 | 30 rows | sum=5.0 | max=2.0 | earliest=2022-03-20T21:00:00+08:00  
  3. checkoutservice-2 | 29 rows | sum=2.0 | max=1.0 | earliest=2022-03-20T21:00:00+08:00  
  4. frontend-0 | 31 rows | sum=1.0 | max=1.0 | earliest=2022-03-20T21:00:00+08:00  
  5. frontend-2 | 30 rows | sum=1.0 | max=1.0 | earliest=2022-03-20T21:00:00+08:00  
  6. checkoutservice-0 | 30 rows | sum=1.0 | max=1.0 | earliest=2022-03-20T21:00:00+08:00  
- Many other services (cartservice-*, currencyservice-*, adservice-*, emailservice-*, paymentservice-*) reported zero summed error_count in-window (present but sum=0).

Interpretation: frontend-1 and checkoutservice-1 show the highest aggregated error counts and should be prioritized for investigation.

2) error_logs.csv — distinct samples (ordered by timestamp, up to 20) and counts per cmdb_id
- Distinct error log samples (timestamp, cmdb_id, message) — earliest → latest:
  - 1647782252 | frontend-1: "severity: error, message: request error"  
  - 1647782252 | checkoutservice-1: "severity: warning, message: failed to send ord..."  
  - 1647782258 | frontend-2: "severity: error, message: request error"  
  - 1647782258 | checkoutservice-1: "severity: warning, message: failed to send ord..."  
  - 1647782281 | frontend-1: "severity: error, message: request error"  
  - 1647782281 | checkoutservice-2: "severity: warning, message: failed to send ord..."  
  - 1647782346 | frontend-1: "severity: error, message: request error"  
  - 1647782346 | checkoutservice-2: "severity: warning, message: failed to send ord..."  
  - 1647782400 | checkoutservice-1: "severity: warning, message: failed to send ord..."  
  - 1647782400 | frontend-1: "severity: error, message: request error"  
  - 1647782407 | frontend-1: "severity: error, message: request error"  
  - 1647782407 | checkoutservice-0: "severity: warning, message: failed to send ord..."  
  - 1647782415 | checkoutservice-1: "severity: warning, message: failed to send ord..."  
  - 1647782415 | frontend-0: "severity: error, message: request error"  
  - 1647782465 | checkoutservice-1: "severity: warning, message: failed to send ord..."  
  - 1647782465 | frontend-1: "severity: error, message: request error"

- Error-log counts per cmdb_id (top 10):
  1. frontend-1: 6  
  2. checkoutservice-1: 5  
  3. checkoutservice-2: 2  
  4. checkoutservice-0: 1  
  5. frontend-0: 1  
  6. frontend-2: 1

Interpretation: error_logs corroborate the log.error_count aggregates — frontend-1 and checkoutservice-1 show the most frequent error/warning entries. Messages show repeated "request error" (frontend) and "failed to send ord..." warnings (checkoutservice), occurring multiple times across the window.

Actionable next steps (suggested)
- Prioritize investigating frontend-1 and checkoutservice-1 (and checkoutservice-2) — they show the highest error counts and error-log occurrences.
- Inspect request traces and network/connection errors between frontend and checkoutservice components; review recent deployments or resource issues on those instances.
- For other services with sum=0 but present, deprioritize unless further trace/metric evidence appears.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  count_of_rows_in_window  sum_value_in_window  max_value_in_window       earliest_ts_shanghai
0          frontend-1                       31                  6.0                  2.0  2022-03-20T21:00:00+08:00
1   checkoutservice-1                       30                  5.0                  2.0  2022-03-20T21:00:00+08:00
2   checkoutservice-2                       29                  2.0                  1.0  2022-03-20T21:00:00+08:00
3          frontend-0                       31                  1.0                  1.0  2022-03-20T21:00:00+08:00
4          frontend-2                       30                  1.0                  1.0  2022-03-20T21:00:00+08:00
5   checkoutservice-0                       30                  1.0                  1.0  2022-03-20T21:00:00+08:00
6         adservice-2                       30                  0.0                  0.0  2022-03-20T21:00:00+08:00
7       cartservice-2                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
8       cartservice-1                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
9   currencyservice-0                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
10  currencyservice-1                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
11  currencyservice-2                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
12      cartservice-0                       31                  0.0                  0.0  2022-03-20T21:00:00+08:00
13        adservice-1                       30                  0.0                  0.0  2022-03-20T21:00:00+08:00
14        adservice-0                       30                  0.0                  0.0  2022-03-20T21:00:00+08:00
15     emailservice-2                       30                  0.0                  0.0  2022-03-20T21:00:00+08:00
16     emailservice-1                       28                  0.0                  0.0  2022-03-20T21:00:00+08:00
17     emailservice-0                       29                  0.0                  0.0  2022-03-20T21:00:00+08:00
18   paymentservice-0                       26                  0.0                  0.0  2022-03-20T21:00:00+08:00
19   paymentservice-1                       29                  0.0                  0.0  2022-03-20T21:00:00+08:00,      timestamp            cmdb_id                                            message
0   1647782252         frontend-1            severity: error, message: request error
1   1647782252  checkoutservice-1  severity: warning, message: failed to send ord...
2   1647782258         frontend-2            severity: error, message: request error
3   1647782258  checkoutservice-1  severity: warning, message: failed to send ord...
4   1647782281         frontend-1            severity: error, message: request error
5   1647782281  checkoutservice-2  severity: warning, message: failed to send ord...
6   1647782346         frontend-1            severity: error, message: request error
7   1647782346  checkoutservice-2  severity: warning, message: failed to send ord...
8   1647782400  checkoutservice-1  severity: warning, message: failed to send ord...
9   1647782400         frontend-1            severity: error, message: request error
10  1647782407         frontend-1            severity: error, message: request error
11  1647782407  checkoutservice-0  severity: warning, message: failed to send ord...
12  1647782415  checkoutservice-1  severity: warning, message: failed to send ord...
13  1647782415         frontend-0            severity: error, message: request error
14  1647782465  checkoutservice-1  severity: warning, message: failed to send ord...
15  1647782465         frontend-1            severity: error, message: request error,              cmdb_id  error_count_in_window
0         frontend-1                      6
1  checkoutservice-1                      5
2  checkoutservice-2                      2
3  checkoutservice-0                      1
4         frontend-0                      1
5         frontend-2                      1)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import re

# reuse existing df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

# ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)

# timezone
tz = pytz.timezone('Asia/Shanghai')

# target cmdb_ids (deduped)
targets = ['frontend-1','frontend-0','frontend-2','checkoutservice-0','checkoutservice-1','checkoutservice-2']
targets = list(dict.fromkeys(targets))  # preserve order, dedupe

# incident window
start_ts = 1647781200
end_ts = 1647783000

# 1) Filter metric rows for target cmdb_ids and kpi_name containing 'container.node-'
df_targets = df_metric[df_metric['cmdb_id'].isin(targets)].copy()
df_container_kpis = df_targets[df_targets['kpi_name'].str.contains('container.node-', na=False)].copy()

# 2) Extract node_id token from kpi_name using regex 'container.node-X' -> X
# Use regex to capture digits after container.node-
df_container_kpis['node_id'] = df_container_kpis['kpi_name'].str.extract(r'container\.node-(\d+)', expand=False)

# keep distinct (cmdb_id, node_id, kpi_name) combinations
unique_kpis = df_container_kpis[['cmdb_id','node_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# For each (cmdb_id, node_id) pick one sample kpi_name (first) and compute earliest timestamp in the incident window
rows = []
for (cmdb, node), grp in unique_kpis.groupby(['cmdb_id','node_id']):
    # sample kpi_name (take first)
    sample_kpi = grp['kpi_name'].iloc[0]
    # find earliest timestamp in-window for any kpi_name containing this node_id for this cmdb_id
    mask_window = (
        (df_metric['cmdb_id'] == cmdb) &
        (df_metric['timestamp'] >= start_ts) &
        (df_metric['timestamp'] <= end_ts) &
        (df_metric['kpi_name'].str.contains(rf'container\.node-{re.escape(str(node))}', na=False))
    )
    window_rows = df_metric[mask_window]
    if window_rows.empty:
        earliest_iso = None
    else:
        earliest_ts = int(window_rows['timestamp'].min())
        earliest_iso = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    rows.append((cmdb, node, sample_kpi, earliest_iso))

mapping_df = pd.DataFrame(rows, columns=['cmdb_id','node_id','sample_kpi_name','earliest_ts_in_window'])

# Limit to at most 20 rows and only the requested cmdb_ids (already filtered)
mapping_result = mapping_df.head(20)

# Keep intermediates in kernel memory:
df_targets, df_container_kpis, unique_kpis, mapping_df

# Display compact mapping table
mapping_result

```
Out[5]:
```
summary = (
    "Mapping of target instances to node IDs (from kpi names containing 'container.node-'):\n\n"
    "- checkoutservice-0 -> node_id 6 (sample kpi: container.node-6.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n"
    "- checkoutservice-1 -> node_id 6 (sample kpi: container.node-6.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n"
    "- checkoutservice-2 -> node_id 5 (sample kpi: container.node-5.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n"
    "- frontend-0        -> node_id 6 (sample kpi: container.node-6.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n"
    "- frontend-1        -> node_id 5 (sample kpi: container.node-5.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n"
    "- frontend-2        -> node_id 5 (sample kpi: container.node-5.container_cpu_cfs_periods), earliest in-window: 2022-03-20T21:00:00+08:00\n\n"
    "Notes: All listed node-related kpi samples appear at or after the start of the incident window (earliest sample = 2022-03-20T21:00:00+08:00)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id node_id                             sample_kpi_name      earliest_ts_in_window
0  checkoutservice-0       6  container.node-6.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00
1  checkoutservice-1       6  container.node-6.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00
2  checkoutservice-2       5  container.node-5.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00
3         frontend-0       6  container.node-6.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00
4         frontend-1       5  container.node-5.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00
5         frontend-2       5  container.node-5.container_cpu_cfs_periods  2022-03-20T21:00:00+08:00```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse existing df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

# ensure correct dtypes
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)

tz = pytz.timezone('Asia/Shanghai')

# 1) Select relevant series: kpi_name contains container.node-5 or container.node-4
#    OR kpi_name startswith 'system.' and cmdb_id in ['node-5','node-4']
mask_container_5 = df_metric['kpi_name'].str.contains(r'container\.node-5', na=False)
mask_container_4 = df_metric['kpi_name'].str.contains(r'container\.node-4', na=False)
mask_system_node = df_metric['kpi_name'].str.startswith('system.', na=False) & df_metric['cmdb_id'].isin(['node-5','node-4'])

mask_nodes = mask_container_5 | mask_container_4 | mask_system_node
df_nodes = df_metric[mask_nodes].copy()

# 2) Global statistics per (cmdb_id, kpi_name) BEFORE window filtering
group_keys = ['cmdb_id','kpi_name']
gb_nodes = df_nodes.groupby(group_keys)['value']
global_stats_nodes = gb_nodes.agg(
    global_count=('count'),
    global_P95=(lambda x: x.quantile(0.95)),
    global_P5=(lambda x: x.quantile(0.05))
).reset_index()

# 3) Filter to incident window and compute windowed stats
start_ts = 1647781200
end_ts = 1647783000
window_nodes = df_nodes[(df_nodes['timestamp'] >= start_ts) & (df_nodes['timestamp'] <= end_ts)].copy()

window_stats_nodes = window_nodes.groupby(group_keys).agg(
    count_in_window=('value','count'),
    min_in_window=('value','min'),
    max_in_window=('value','max'),
    earliest_timestamp_in_window=('timestamp','min')
).reset_index()

# 4) Mark anomalies in window rows using global thresholds
window_with_thresholds = window_nodes.merge(
    global_stats_nodes[['cmdb_id','kpi_name','global_P95','global_P5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)
window_with_thresholds['is_anomaly'] = (
    (window_with_thresholds['value'] >= window_with_thresholds['global_P95']) |
    (window_with_thresholds['value'] <= window_with_thresholds['global_P5'])
)

# 5) Per-pair anomaly aggregations
anomaly_agg_nodes = window_with_thresholds[window_with_thresholds['is_anomaly']].groupby(group_keys).agg(
    anomaly_count_in_window=('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window=('timestamp','min')
).reset_index()

# Merge globals + window stats + anomaly stats
agg_nodes = global_stats_nodes.merge(window_stats_nodes, on=group_keys, how='left')
agg_nodes = agg_nodes.merge(anomaly_agg_nodes, on=group_keys, how='left')

# Fill missing values for counts
agg_nodes['count_in_window'] = agg_nodes['count_in_window'].fillna(0).astype(int)
agg_nodes['anomaly_count_in_window'] = agg_nodes['anomaly_count_in_window'].fillna(0).astype(int)
agg_nodes['earliest_anomaly_timestamp_in_window'] = agg_nodes['earliest_anomaly_timestamp_in_window'].where(
    ~agg_nodes['earliest_anomaly_timestamp_in_window'].isna(), None)

# Extract node_id for each kpi_name: first attempt from 'container.node-X', else if cmdb_id is node-5/node-4 use that
def extract_node_id(row):
    k = row['kpi_name']
    m = re.search(r'container\.node-(\d+)', k)
    if m:
        return m.group(1)
    # fallback: if cmdb_id is 'node-5' or 'node-4', return that numeric id
    if row['cmdb_id'] in ['node-5','node-4']:
        return row['cmdb_id'].split('-')[-1]
    return None

agg_nodes['node_id'] = agg_nodes.apply(extract_node_id, axis=1)

# 5b) Compute severity
def compute_severity(row):
    p95 = row['global_P95']
    p5 = row['global_P5']
    max_w = row['max_in_window'] if not pd.isna(row['max_in_window']) else np.nan
    min_w = row['min_in_window'] if not pd.isna(row['min_in_window']) else np.nan
    if pd.isna(max_w) and pd.isna(min_w):
        return 0.0
    a = (max_w - p95) / p95 if (pd.notna(p95) and p95 > 0 and pd.notna(max_w)) else 0.0
    b = (p5 - min_w) / p5 if (pd.notna(p5) and p5 > 0 and pd.notna(min_w)) else 0.0
    return float(max(a, b, 0.0))

agg_nodes['severity'] = agg_nodes.apply(compute_severity, axis=1)

# Convert earliest anomaly timestamps to Asia/Shanghai ISO strings
def to_shanghai_iso(ts):
    if ts is None or pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

agg_nodes['earliest_anomaly_ts_shanghai'] = agg_nodes['earliest_anomaly_timestamp_in_window'].apply(to_shanghai_iso)
agg_nodes['earliest_window_ts_shanghai'] = agg_nodes['earliest_timestamp_in_window'].apply(
    lambda x: to_shanghai_iso(x) if not pd.isna(x) else None
)

# 6A) Prepare top-20 table sorted by anomaly_count_in_window desc then severity desc
final_cols = [
    'cmdb_id','kpi_name',
    'global_count','global_P95','global_P5',
    'count_in_window','anomaly_count_in_window','earliest_anomaly_ts_shanghai',
    'min_in_window','max_in_window','severity'
]
top20_nodes = agg_nodes[final_cols].sort_values(
    by=['anomaly_count_in_window','severity'], ascending=[False, False]
).head(20).reset_index(drop=True)

# 6B) Small aggregate for node-5 and node-4:
# Count distinct (cmdb_id,kpi_name) with anomaly_count_in_window>0 for node_id '5' and '4'
# and earliest anomaly timestamp across all flagged KPIs for that node_id
agg_with_node = agg_nodes.copy()
# Ensure node_id present
agg_with_node['node_id'] = agg_with_node['node_id'].astype(object)

node_aggregates = []
for nid in ['5','4']:
    df_n = agg_with_node[agg_with_node['node_id'] == nid]
    flagged = df_n[df_n['anomaly_count_in_window'] > 0]
    count_flagged = int(flagged.shape[0])
    if not flagged.empty and flagged['earliest_anomaly_timestamp_in_window'].notna().any():
        earliest_ts = int(flagged['earliest_anomaly_timestamp_in_window'].min())
        earliest_iso = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    else:
        earliest_iso = None
    node_aggregates.append((f'node-{nid}', count_flagged, earliest_iso))

node_agg_df = pd.DataFrame(node_aggregates, columns=['node','distinct_flagged_kpi_pairs','earliest_anomaly_ts_shanghai'])

# Keep intermediates in kernel memory
df_nodes, global_stats_nodes, window_nodes, window_with_thresholds, anomaly_agg_nodes, agg_nodes

# Return compact results: top20_nodes (<=20 rows) and node_agg_df (two-row aggregate)
top20_nodes, node_agg_df

```
Out[6]:
```
Summary — node-level anomaly scan for node-5 and node-4 (incident window 2022-03-20 21:00–21:30 Asia/Shanghai)

Key findings (top examples):
- checkoutservice-2 | container.node-5.container_fs_reads_MB./dev/vda  
  - global_P95 = 0.01171875, global_P5 = 0.0  
  - window min = 0.0, window max = 0.2617188, anomaly_count_in_window = 31 (all samples flagged)  
  - severity ≈ 21.33 (large relative spike vs global P95)
- checkoutservice-2 | container.node-5.container_fs_reads./dev/vda  
  - global_P95 = 2.0, window max = 22.0, anomaly_count_in_window = 31, severity = 10.0
- node-4 | system.io.svctm  
  - global_P95 = 0.07, window max = 0.5, anomaly_count_in_window = 31, severity ≈ 6.14
- node-5 | system.cpu.iowait  
  - global_P95 = 0.02, window max = 0.13, anomaly_count_in_window = 31, severity ≈ 5.50

Additional context:
- Many node-5 related container KPIs (e.g., file descriptors, sockets, memory, CPU throttle metrics) were flagged; some have modest severity but consistent anomalies across the whole window.
- Several series have global_P5 = 0 causing lower-bound flags; this can inflate anomaly counts for zero-valued baselines (threshold artifact). Still, large positive spikes (high severity) are present for I/O and CPU-wait metrics.

Compact aggregates (per node):
- node-5: 440 distinct (cmdb_id,kpi_name) pairs had anomaly_count_in_window > 0. Earliest anomaly observed: 2022-03-20T21:00:00+08:00.
- node-4: 56 distinct (cmdb_id,kpi_name) pairs had anomaly_count_in_window > 0. Earliest anomaly observed: 2022-03-20T21:00:00+08:00.

Interpretation and next steps:
- Node-5 shows substantially more flagged KPIs — prioritize investigation on node-5 first. Focus on disk I/O and filesystem read/write metrics and CPU iowait (these show highest severities), and on services mapped to node-5 (e.g., checkoutservice-2).
- Validate anomalies where global quantiles are zero (could be threshold artifact). For high-severity metrics (large severity values) treat them as real spikes and inspect disk/host performance, task scheduling, and recent deployments on node-5/node-4.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  global_count    global_P95     global_P5  count_in_window  anomaly_count_in_window earliest_anomaly_ts_shanghai  min_in_window  max_in_window   severity
0   checkoutservice-2    container.node-5.container_fs_reads_MB./dev/vda            61  1.171875e-02  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   2.617188e-01  21.333333
1   checkoutservice-2       container.node-5.container_fs_reads./dev/vda            61  2.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   2.200000e+01  10.000000
2              node-4                                    system.io.svctm            61  7.000000e-02  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   5.000000e-01   6.142857
3              node-5                                  system.cpu.iowait            61  2.000000e-02  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   1.300000e-01   5.500000
4      cartservice2-0        container.node-5.container_file_descriptors            61  1.630000e+02  1.620000e+02               31                       31    2022-03-20T21:00:00+08:00   1.225000e+02   1.630000e+02   0.243827
5      cartservice2-0                 container.node-5.container_sockets            61  8.000000e+00  7.000000e+00               31                       31    2022-03-20T21:00:00+08:00   5.750000e+00   8.000000e+00   0.178571
6              node-5                             system.udp.connect.num            61  1.100000e+01  1.100000e+01               31                       31    2022-03-20T21:00:00+08:00   1.100000e+01   1.200000e+01   0.090909
7         adservice-2                 container.node-5.container_threads            61  2.700000e+01  2.700000e+01               31                       31    2022-03-20T21:00:00+08:00   2.700000e+01   2.766667e+01   0.024691
8   checkoutservice-2            container.node-5.container_memory_cache            61  6.778880e+06  6.508544e+06               31                       31    2022-03-20T21:00:00+08:00   6.508544e+06   6.914048e+06   0.019940
9         adservice-2   container.node-5.container_memory_working_set_MB            61  1.565820e+02  1.565352e+02               31                       31    2022-03-20T21:00:00+08:00   1.565352e+02   1.566107e+02   0.000183
10        adservice-2         container.node-5.container_memory_usage_MB            61  1.595586e+02  1.595117e+02               31                       31    2022-03-20T21:00:00+08:00   1.595117e+02   1.595872e+02   0.000180
11        adservice-2  container.node-5.container_cpu_cfs_throttled_p...            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
12        adservice-2  container.node-5.container_cpu_cfs_throttled_s...            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
13        adservice-2    container.node-5.container_cpu_load_average_10s            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
14        adservice-2     container.node-5.container_fs_inodes./dev/vda1            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
15        adservice-2  container.node-5.container_fs_inodes_free./dev...            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
16        adservice-2  container.node-5.container_fs_io_current./dev/...            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
17        adservice-2  container.node-5.container_fs_io_time_seconds....            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
18        adservice-2  container.node-5.container_fs_io_time_weighted...            61  0.000000e+00  0.000000e+00               31                       31    2022-03-20T21:00:00+08:00   0.000000e+00   0.000000e+00   0.000000
19        adservice-2   container.node-5.container_fs_limit_MB./dev/vda1            61  6.046307e+05  6.046307e+05               31                       31    2022-03-20T21:00:00+08:00   6.046307e+05   6.046307e+05   0.000000,      node  distinct_flagged_kpi_pairs earliest_anomaly_ts_shanghai
0  node-5                         440    2022-03-20T21:00:00+08:00
1  node-4                          56    2022-03-20T21:00:00+08:00)```
```